# 02 · Preprocessing

Turns the raw recordings into the exact tensors the model trains on, and freezes the train/val/test
split so every later notebook sees the same data.

Each recording is:

1. decoded and resampled to **16 kHz mono**,
2. **silence trimmed** (`audio.trim`),
3. **loudness normalised** to a target RMS (`audio.normalize`),
4. **fitted to `audio.clip_seconds`** by padding or cropping,
5. written as **PCM16 WAV** under `processed/<class>/`.

The result is `processed/manifest.csv` — the single input of notebooks 03, 04 and 05.

Re-running is cheap: existing files are skipped unless `OVERWRITE = True`.

In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


## 1 · Index the dataset and load the validation report

If `01_dataset.ipynb` has been run, its report tells us which files to exclude. Otherwise a quick
header-only validation is run here.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.dataset import load_phrases, scan_dataset, validate_dataset

paths = config.paths
phrases = load_phrases(paths.phrases_path)
index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
)
print("indexed %d recordings across %d classes" % (len(index), index.num_classes))

report_path = paths.reports_path / "validation_report.json"
BLOCKING = {"corrupted", "empty", "silent"}

if report_path.is_file():
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    unusable = sorted({item["path"] for item in payload["issues"] if item["kind"] in BLOCKING})
    print("loaded validation report from %s" % report_path)
else:
    print("no validation report found — running a quick header-only validation")
    quick = validate_dataset(index, config.audio, deep=False)
    unusable = quick.unusable_paths()

print("%d recording(s) excluded as unusable" % len(unusable))


## 2 · Write conditioned copies

Output goes to `processed/` on Drive. Set `OVERWRITE = True` after changing any `audio.*` setting —
otherwise clips written with the old settings are kept.

In [ ]:
from src.dataset import preprocess_dataset

OVERWRITE = False

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

records, summary = preprocess_dataset(
    index,
    config,
    exclude=unusable,
    overwrite=OVERWRITE,
    progress=progress,
)
print()
print(summary.summary())
print("clips in manifest:", len(records))

if summary.failed:
    print()
    print("failed files are listed above and are absent from the manifest")


## 3 · Split into train / val / test

Stratified per class, so every class keeps its proportion in each split. Ratios come from
`split.*` in the config, and the seed makes the split reproducible.

Set `split.group_regex` in the config when several recordings share a speaker — whole groups then
move together, which stops a speaker appearing in both train and test.

In [ ]:
from src.dataset import assign_splits, save_manifest, split_counts

records = assign_splits(records, config.split, config.seed)
counts = split_counts(records)
print("split sizes:", counts)

frame = pd.DataFrame([
    {"class": record.label, "split": record.split} for record in records
])
pivot = frame.pivot_table(index="class", columns="split", aggfunc=len, fill_value=0)
display(pivot)

manifest_path = save_manifest(records, paths.manifest_path)
print()
print("manifest written to", manifest_path)


## 4 · Preview the front-end

What the model actually sees: a `(frames, mel bins)` log mel spectrogram. Shape and framing come
from `features.*`, and this exact geometry is what the exported TFLite model expects.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio, read_wav
from src.features import LogMelExtractor
from src import visualization as viz

extractor = LogMelExtractor(config.features, config.audio.sample_rate)
frames, mel_bins, channels = config.input_shape
print("model input: %d frames x %d mel bins x %d channel" % (frames, mel_bins, channels))

rng = np.random.default_rng(config.seed)
previews, titles = [], []
for label in sorted({record.label for record in records}):
    for_class = [record for record in records if record.label == label]
    chosen = for_class[int(rng.integers(len(for_class)))]
    clip, _ = read_wav(chosen.resolve(paths.processed_path))
    previews.append(extractor(clip))
    titles.append("%s · %s" % (label, Path(chosen.path).name))

figure = viz.plot_feature_grid(previews[:9], titles[:9], hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_feature_previews.png")

example = records[int(rng.integers(len(records)))]
raw = load_audio(example.source_path, config.audio.sample_rate)
conditioned, _ = read_wav(example.resolve(paths.processed_path))
print()
print("before / after conditioning:", Path(example.source_path).name)
display(Audio(raw, rate=config.audio.sample_rate))
display(Audio(conditioned, rate=config.audio.sample_rate))
viz.save_figure(
    viz.plot_waveform(conditioned, config.audio.sample_rate, title="conditioned clip"),
    paths.reports_path / "02_conditioned_waveform.png",
)


## 5 · Preview the augmentation

Augmentation runs **on the fly during training only** — nothing here is written to Drive. This cell
shows what each transform does to one clip so the ranges in `augmentation.*` can be sanity checked
by eye and by ear.

If `noise/` on Drive is empty, background noise falls back to synthetic white/pink noise. Dropping
real room recordings in there is the single cheapest accuracy win for a phone-deployed model.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor, spec_augment

noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
print("noise clips available:", len(noise_bank))

augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)
base, _ = read_wav(records[0].resolve(paths.processed_path))

# Force one transform at a time by disabling the others.
def only(name: str):
    single = config.with_overrides({
        "augmentation.%s.probability" % other: 0.0
        for other in ["background_noise", "pitch_shift", "speed_perturb", "gain", "time_shift"]
        if other != name
    }).with_overrides({"augmentation.%s.probability" % name: 1.0})
    return WaveformAugmentor(single.augmentation, single.audio, noise_bank)

variants = [("original", base)]
for name in ["background_noise", "pitch_shift", "speed_perturb", "gain", "time_shift"]:
    variants.append((name, only(name)(base, np.random.default_rng(config.seed))))

for name, clip in variants:
    print(name)
    display(Audio(clip, rate=config.audio.sample_rate))

panels = [extractor(clip) for _, clip in variants]
labels = [name for name, _ in variants]
panels.append(spec_augment(
    extractor(base),
    config.with_overrides({"augmentation.spec_augment.probability": 1.0}).augmentation,
    np.random.default_rng(config.seed),
))
labels.append("spec_augment")

figure = viz.plot_feature_grid(panels, labels, hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_augmentation_previews.png")


## 6 · Global feature statistics (optional)

Only needed when `features.normalize` is `global`. The default, `per_example`, normalises each clip
on its own and needs no dataset statistics — which also makes the Android port simpler.

In [ ]:
from src.features import compute_global_stats

if config.features.normalize == "global":
    train_records = [record for record in records if record.split == "train"]
    sample_records = train_records[: min(len(train_records), 2000)]
    raw_extractor = LogMelExtractor(
        config.with_overrides({"features.normalize": "none"}).features,
        config.audio.sample_rate,
    )
    stats = compute_global_stats(
        raw_extractor(read_wav(record.resolve(paths.processed_path))[0])
        for record in sample_records
    )
    stats_path = stats.save(paths.processed_path / "feature_stats.json")
    print("global stats over %d clips:" % len(sample_records), stats)
    print("written to", stats_path)
else:
    print("features.normalize = %r — no global statistics needed" % config.features.normalize)


## 7 · Done

Written to Drive:

* `processed/<class>/*.wav` — conditioned 16 kHz mono PCM16 clips
* `processed/manifest.csv` — path, class, phrase id, split for every clip
* `reports/02_*.png` — front-end and augmentation previews

Continue with `03_training.ipynb`.

In [ ]:
print("processed clips :", len(records))
print("splits          :", split_counts(records))
print("manifest        :", paths.manifest_path)
print("processed root  :", paths.processed_path)
